# ER-CyRIS Algorithmic Diversity Benchmark

**Tujuan.** Notebook ini menguji apakah bukti pada modul Robust Detection,
Explanation Reliability, dan Dual-Domain Reliability Gate tetap relevan ketika
keluarga algoritma berubah. Notebook ini tidak mengulang eksperimen H1-H5 dan
tidak mengubah hasil H4 yang sudah dibekukan.

**Keluarga algoritma.** Linear, single tree, bagging ensemble, boosting,
kernel, dan neural network.

**Batas klaim.** Hasil notebook ini hanya mendukung algorithmic-diversity
benchmark. Risk Translation dan Governed Response memerlukan scenario analysis,
outcome risiko, dan validasi pakar yang terpisah. Holdout pada notebook ini
adalah *untouched internal holdout*, bukan external organizational validation.


## Urutan eksekusi wajib

1. Jalankan instalasi, konfigurasi, loader, dan audit data.
2. Jalankan development benchmark sampai selesai.
3. Tinjau tabel hasil dan bekukan threshold.
4. Pastikan berkas `FROZEN_THRESHOLD_REGISTRY.json` sudah terbentuk.
5. Ubah `RUN_UNTOUCHED_HOLDOUT=True` hanya setelah threshold dibekukan.
6. Jalankan holdout satu kali. Notebook memakai lock dan resume per-run.

Jangan memilih model atau threshold berdasarkan hasil holdout.


In [14]:
%pip install -q xgboost shap imbalanced-learn pyarrow

import gc
import hashlib
import inspect
import itertools
import json
import math
import os
import re
import resource
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, spearmanr, wilcoxon
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
print("Imports ready")


Imports ready


In [16]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE = Path('/content/drive/MyDrive/ercyris_siklus2') if IN_COLAB else Path('.')
OUT_DIR = (
    Path('/content/drive/MyDrive/ercyris_framework_validation/algorithmic_diversity_v1')
    if IN_COLAB else Path('algorithmic_diversity_v1')
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'predictions_development').mkdir(exist_ok=True)
(OUT_DIR / 'predictions_holdout').mkdir(exist_ok=True)

PATHS = {
    'BGL': BASE / 'BGL.log',
    'HDFS': BASE / 'HDFS.log',
    'HDFS_LABEL': BASE / 'anomaly_label.csv',
    'UNSW-NB15': BASE / 'UNSW-NB15_1.csv',
    'CICIDS2018': BASE / '02-14-2018.csv',
}

PRIMARY_DATASETS = ['BGL', 'HDFS', 'UNSW-NB15']
INCLUDE_CICIDS_CONTRAST = False
SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ['raw_telemetry_space', 'transformed_feature_space']
CONDITIONS = {'clean': 0.0, 'noise_sigma_0p05': 0.05}
MAX_SAMPLES_PER_DATASET = 30_000
BGL_PARSE_LIMIT = 250_000

RUN_DEVELOPMENT = True
RUN_XAI = True
XAI_SEEDS = [42]
SHAP_N = 80
SHAP_BACKGROUND_N = 40
TOP_K = 10

# Tetap False sampai threshold registry selesai ditinjau dan dibekukan.
RUN_UNTOUCHED_HOLDOUT = True

print('Output:', OUT_DIR)
print('Primary datasets:', PRIMARY_DATASETS)
print('CICIDS secondary contrast:', INCLUDE_CICIDS_CONTRAST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output: /content/drive/MyDrive/ercyris_framework_validation/algorithmic_diversity_v1
Primary datasets: ['BGL', 'HDFS', 'UNSW-NB15']
CICIDS secondary contrast: False


## Utilitas yang dikunci

Empat partisi digunakan: 55% train, 15% calibration, 15% development,
dan 15% untouched holdout. Calibration dan threshold selection tidak pernah
memakai holdout.


In [18]:
"""Core utilities for the ER-CyRIS algorithmic-diversity benchmark."""

from __future__ import annotations

import hashlib
import json

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def _as_binary_vector(y) -> np.ndarray:
    values = np.asarray(y, dtype=int).reshape(-1)
    if len(values) < 8:
        raise ValueError("At least eight observations are required")
    if set(np.unique(values)) != {0, 1}:
        raise ValueError("Binary labels must contain both 0 and 1")
    return values


def make_four_way_split(y, seed: int = 42) -> dict[str, np.ndarray]:
    """Return disjoint 55/15/15/15 stratified index partitions."""
    labels = _as_binary_vector(y)
    indices = np.arange(len(labels))
    train_idx, remainder_idx = train_test_split(
        indices,
        train_size=0.55,
        random_state=seed,
        stratify=labels,
    )
    middle_idx, holdout_idx = train_test_split(
        remainder_idx,
        test_size=1.0 / 3.0,
        random_state=seed + 1,
        stratify=labels[remainder_idx],
    )
    calibration_idx, development_idx = train_test_split(
        middle_idx,
        test_size=0.5,
        random_state=seed + 2,
        stratify=labels[middle_idx],
    )
    return {
        "train": np.sort(train_idx),
        "calibration": np.sort(calibration_idx),
        "development": np.sort(development_idx),
        "holdout": np.sort(holdout_idx),
    }


def add_gaussian_noise(X, sigma: float, seed: int) -> np.ndarray:
    values = np.asarray(X, dtype=float)
    if sigma < 0:
        raise ValueError("sigma must be non-negative")
    if sigma == 0:
        return values.copy()
    noise = np.random.default_rng(seed).normal(0.0, sigma, size=values.shape)
    return np.clip(values + noise, 0.0, 1.0)


def expected_calibration_error(y_true, probabilities, n_bins: int = 10) -> float:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")
    if n_bins < 1:
        raise ValueError("n_bins must be positive")

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.minimum(np.digitize(probs, edges[1:-1], right=False), n_bins - 1)
    ece = 0.0
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        accuracy = float(labels[mask].mean())
        confidence = float(probs[mask].mean())
        ece += float(mask.mean()) * abs(accuracy - confidence)
    return float(ece)


def select_f1_threshold(y_true, probabilities) -> tuple[float, float]:
    labels = np.asarray(y_true, dtype=int).reshape(-1)
    probs = np.clip(np.asarray(probabilities, dtype=float).reshape(-1), 0.0, 1.0)
    if len(labels) != len(probs) or len(labels) == 0:
        raise ValueError("Labels and probabilities must have the same non-zero length")

    candidates = np.unique(probs)
    scored = [
        (float(f1_score(labels, probs >= threshold, zero_division=0)), float(threshold))
        for threshold in candidates
    ]
    best_score = max(score for score, _ in scored)
    tied = [threshold for score, threshold in scored if score == best_score]
    best_threshold = min(tied, key=lambda value: (abs(value - 0.5), value))
    return float(best_threshold), float(best_score)


class PlattCalibrator:
    """One-dimensional sigmoid calibration fitted only on the calibration split."""

    def __init__(self, seed: int = 42):
        self.seed = int(seed)
        self.model = LogisticRegression(random_state=self.seed, solver="lbfgs")
        self._fitted = False

    def fit(self, scores, y):
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        labels = np.asarray(y, dtype=int).reshape(-1)
        if len(score_array) != len(labels) or set(np.unique(labels)) != {0, 1}:
            raise ValueError("Calibration data must contain aligned binary classes")
        self.model.fit(score_array, labels)
        self._fitted = True
        return self

    def predict_proba(self, scores) -> np.ndarray:
        if not self._fitted:
            raise RuntimeError("Call fit before predict_proba")
        score_array = np.asarray(scores, dtype=float).reshape(-1, 1)
        return self.model.predict_proba(score_array)[:, 1]


def stratified_cap_indices(y, max_samples: int, seed: int) -> np.ndarray:
    labels = _as_binary_vector(y)
    if max_samples < 2:
        raise ValueError("max_samples must be at least two")
    if len(labels) <= max_samples:
        return np.arange(len(labels), dtype=int)
    chosen, _ = train_test_split(
        np.arange(len(labels)),
        train_size=int(max_samples),
        random_state=seed,
        stratify=labels,
    )
    return np.sort(chosen)


def expected_run_count(
    n_datasets: int,
    n_models: int,
    n_representations: int,
    n_seeds: int,
    n_conditions: int,
) -> dict[str, int]:
    model_fits = int(n_datasets * n_models * n_representations * n_seeds)
    return {
        "model_fits": model_fits,
        "evaluation_rows": int(model_fits * n_conditions),
    }


def build_run_id(
    dataset: str,
    representation: str,
    model: str,
    seed: int,
    condition: str,
    split: str,
) -> str:
    return (
        f"{dataset}__{representation}__{model}__seed-{int(seed)}"
        f"__{condition}__{split}"
    )


def freeze_registry_digest(records: list[dict]) -> str:
    canonical_records = sorted(
        (json.dumps(record, sort_keys=True, separators=(",", ":")) for record in records)
    )
    payload = "\n".join(canonical_records).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


In [19]:
class BaselineRepresentation:
    '''Median imputation plus min-max scaling fitted on train only.'''

    name = 'raw_telemetry_space'

    def __init__(self):
        self.imputer = SimpleImputer(strategy='median')
        self.scaler = MinMaxScaler()

    def fit(self, X):
        Xi = self.imputer.fit_transform(np.asarray(X, dtype=float))
        self.scaler.fit(Xi)
        return self

    def transform(self, X):
        Xi = self.imputer.transform(np.asarray(X, dtype=float))
        return np.clip(self.scaler.transform(Xi), 0.0, 1.0)

    def feature_names(self, names):
        return list(names)


class EvidenceAwareRepresentation(BaselineRepresentation):
    '''Variance weighting plus deviation features, rescaled on train only.'''

    name = 'transformed_feature_space'

    def __init__(self):
        super().__init__()
        self.idf = self.mu = self.sd = None
        self.final_scaler = MinMaxScaler()

    def _augment(self, base):
        weighted = base * (1.0 + 0.3 * self.idf)
        deviation = np.clip(np.abs(base - self.mu) / self.sd, 0.0, 5.0) / 5.0
        deviation = deviation[:, :min(5, deviation.shape[1])]
        return np.hstack([weighted, deviation])

    def fit(self, X):
        super().fit(X)
        base = super().transform(X)
        variance = np.var(base, axis=0) + 1e-9
        self.idf = np.log(1.0 / variance + 1.0)
        self.idf /= self.idf.max() + 1e-9
        self.mu = base.mean(axis=0)
        self.sd = base.std(axis=0) + 1e-9
        self.final_scaler.fit(self._augment(base))
        return self

    def transform(self, X):
        base = super().transform(X)
        return np.clip(self.final_scaler.transform(self._augment(base)), 0.0, 1.0)

    def feature_names(self, names):
        base = list(names)
        return base + [f'deviation_{name}' for name in base[:min(5, len(base))]]


REPRESENTATION_FACTORY = {
    'raw_telemetry_space': BaselineRepresentation,
    'transformed_feature_space': EvidenceAwareRepresentation,
}
print('Representations ready:', list(REPRESENTATION_FACTORY))


Representations ready: ['raw_telemetry_space', 'transformed_feature_space']


In [20]:
def stable_bucket(value, modulus=1000):
    digest = hashlib.blake2b(str(value).encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'big') % modulus


def load_bgl(path):
    severity = {
        'INFO': 1, 'DEBUG': 0, 'WARN': 2, 'WARNING': 2, 'ERROR': 3,
        'SEVERE': 4, 'FATAL': 4, 'APPFPO': 3, 'KERN': 2,
        'KERNSVC': 3, 'APPFPW': 2,
    }
    attack_words = [
        'fatal', 'error', 'fail', 'exception', 'abort', 'illegal',
        'segfault', 'panic', 'crash', 'out of memory', 'overflow',
    ]
    rows = []
    with open(path, 'r', errors='replace') as stream:
        for line_number, line in enumerate(stream):
            if BGL_PARSE_LIMIT and line_number >= BGL_PARSE_LIMIT:
                break
            parts = line.split(None, 8)
            if len(parts) < 9:
                continue
            label = 0 if parts[0] == '-' else 1
            sev = severity.get(parts[6].upper(), 1)
            message = parts[8].lower()
            try:
                hour = int(parts[3].split(':')[0]) if ':' in parts[3] else 12
            except Exception:
                hour = 12
            rows.append({
                'severity': sev,
                'is_error': int(sev >= 3),
                'is_warning': int(sev == 2),
                'hour': hour,
                'off_hour': int(hour < 6),
                'peak_hour': int(8 <= hour <= 18),
                'attack_keyword': int(any(word in message for word in attack_words)),
                'has_long_number': int(bool(re.search(r'\b\d{4,}\b', message))),
                'message_length': min(len(message), 500),
                'component_length': len(parts[5]),
                'node_hash': stable_bucket(parts[4]),
                'label': label,
            })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_hdfs(log_path, label_path):
    line_pattern = re.compile(r'^(\d{6})\s+(\d{6})\s+\d+\s+(\w+)\s+([^:]+):\s+(.*)')
    block_pattern = re.compile(r'(blk_-?\d+)')
    severity = {'DEBUG': 0, 'INFO': 1, 'WARN': 2, 'WARNING': 2, 'ERROR': 3, 'FATAL': 4}
    blocks = defaultdict(lambda: {'n': 0, 'error': 0, 'warn': 0, 'components': set(), 'max': 0, 'sum': 0})
    with open(log_path, 'r', errors='replace') as stream:
        for line in stream:
            match = line_pattern.match(line)
            if not match:
                continue
            _, _, level, component, message = match.groups()
            sev = severity.get(level.upper(), 1)
            for block in block_pattern.findall(message):
                row = blocks[block]
                row['n'] += 1
                row['components'].add(component.strip())
                row['max'] = max(row['max'], sev)
                row['sum'] += sev
                row['error'] += int(sev >= 3)
                row['warn'] += int(sev == 2)
    labels = pd.read_csv(label_path)
    label_map = dict(zip(
        labels['BlockId'].astype(str),
        (labels['Label'].str.upper() == 'ANOMALY').astype(int),
    ))
    rows = []
    for block, row in blocks.items():
        if block not in label_map:
            continue
        count = max(row['n'], 1)
        rows.append({
            'n_events': row['n'],
            'n_error': row['error'],
            'n_warn': row['warn'],
            'n_components': len(row['components']),
            'max_severity': row['max'],
            'mean_severity': row['sum'] / count,
            'error_ratio': row['error'] / count,
            'warn_ratio': row['warn'] / count,
            'label': label_map[block],
        })
    frame = pd.DataFrame(rows)
    y = frame.pop('label').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_unsw(path):
    frame = pd.read_csv(path, low_memory=False)
    frame.columns = [column.lower().strip() for column in frame.columns]
    label_column = next((c for c in ['label', 'is_attack'] if c in frame.columns), frame.columns[-1])
    y = (pd.to_numeric(frame[label_column], errors='coerce').fillna(0) != 0).astype(int).to_numpy()
    frame.drop(columns=[label_column, 'srcip', 'dstip', 'attack_cat'], errors='ignore', inplace=True)
    for column in ['proto', 'state', 'service']:
        if column in frame.columns:
            frame[column] = LabelEncoder().fit_transform(frame[column].fillna('unknown').astype(str))
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_cicids(path):
    frame = pd.read_csv(path, low_memory=False)
    label_column = next(column for column in frame.columns if 'label' in column.lower())
    y = (frame[label_column].astype(str).str.strip().str.upper() != 'BENIGN').astype(int).to_numpy()
    frame.drop(columns=[label_column], inplace=True)
    frame.drop(columns=[
        column for column in frame.columns
        if any(key in column.lower() for key in ['timestamp', 'flow id', 'src ip', 'dst ip'])
    ], errors='ignore', inplace=True)
    frame.columns = [column.strip().lower().replace(' ', '_').replace('/', '_per_') for column in frame.columns]
    frame = frame.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    frame['__label__'] = y
    features = [column for column in frame.columns if column != '__label__']
    frame = frame.drop_duplicates(subset=features, keep='first').reset_index(drop=True)
    y = frame.pop('__label__').to_numpy(dtype=int)
    return frame.to_numpy(dtype=float), y, list(frame.columns)


def load_requested_datasets():
    requested = list(PRIMARY_DATASETS)
    if INCLUDE_CICIDS_CONTRAST:
        requested.append('CICIDS2018')
    loaded = {}
    for name in requested:
        if name == 'BGL':
            required = [PATHS['BGL']]
            loader = lambda: load_bgl(PATHS['BGL'])
        elif name == 'HDFS':
            required = [PATHS['HDFS'], PATHS['HDFS_LABEL']]
            loader = lambda: load_hdfs(PATHS['HDFS'], PATHS['HDFS_LABEL'])
        elif name == 'UNSW-NB15':
            required = [PATHS['UNSW-NB15']]
            loader = lambda: load_unsw(PATHS['UNSW-NB15'])
        else:
            required = [PATHS['CICIDS2018']]
            loader = lambda: load_cicids(PATHS['CICIDS2018'])
        missing = [str(path) for path in required if not path.exists()]
        if missing:
            raise FileNotFoundError(f'{name}: missing {missing}')
        X, y, names = loader()
        cap = stratified_cap_indices(y, MAX_SAMPLES_PER_DATASET, seed=42)
        loaded[name] = (X[cap], y[cap], names)
        print(f'{name}: {len(cap):,} rows, {X.shape[1]} features, anomaly={y[cap].mean():.3%}')
    return loaded


In [21]:
MODEL_NAMES = [
    'LogisticRegression',
    'DecisionTree',
    'RandomForest',
    'XGBoost',
    'SVC_RBF',
    'MLP',
]


def make_model(name, seed):
    registry = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000, solver='liblinear', random_state=seed,
        ),
        'DecisionTree': DecisionTreeClassifier(
            min_samples_leaf=2, random_state=seed,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=seed,
        ),
        'XGBoost': XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, tree_method='hist',
            eval_metric='logloss', n_jobs=-1, random_state=seed, verbosity=0,
        ),
        'SVC_RBF': SVC(
            kernel='rbf', probability=False, cache_size=4000, random_state=seed,
        ),
        'MLP': MLPClassifier(
            hidden_layer_sizes=(128, 64), early_stopping=True,
            max_iter=150, batch_size='auto', random_state=seed,
        ),
    }
    return registry[name]


def fit_with_balancing(model, X, y, seed):
    weights = compute_sample_weight(class_weight='balanced', y=y)
    if 'sample_weight' in inspect.signature(model.fit).parameters:
        model.fit(X, y, sample_weight=weights)
        return model
    # Compatibility fallback for estimators without sample_weight.
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(y == label) for label in (0, 1)]
    target = max(len(indices) for indices in class_indices)
    balanced = np.concatenate([
        rng.choice(indices, size=target, replace=len(indices) < target)
        for indices in class_indices
    ])
    rng.shuffle(balanced)
    model.fit(X[balanced], y[balanced])
    return model


def raw_model_score(model, X):
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(X), dtype=float).reshape(-1)
    probability = np.asarray(model.predict_proba(X), dtype=float)[:, 1]
    probability = np.clip(probability, 1e-6, 1.0 - 1e-6)
    return np.log(probability / (1.0 - probability))


print('Models ready:', MODEL_NAMES)


Models ready: ['LogisticRegression', 'DecisionTree', 'RandomForest', 'XGBoost', 'SVC_RBF', 'MLP']


In [22]:
DEVELOPMENT_RESULTS = OUT_DIR / 'development_results.csv'
THRESHOLD_REGISTRY = OUT_DIR / 'threshold_registry.csv'
XAI_RESULTS = OUT_DIR / 'xai_stability_results.csv'
MANIFEST_PATH = OUT_DIR / 'algorithmic_diversity_manifest.json'


def atomic_csv(frame, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def append_unique_row(path, row, key='run_id'):
    current = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if not current.empty and key in current.columns and row[key] in set(current[key].astype(str)):
        return False
    updated = pd.concat([current, pd.DataFrame([row])], ignore_index=True)
    atomic_csv(updated, path)
    return True


def peak_rss_mb():
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return float(value / 1024.0 if value > 10_000 else value)


def classification_metrics(y, probability, threshold):
    probability = np.clip(np.asarray(probability, dtype=float), 0.0, 1.0)
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {
        'precision': precision_score(y, prediction, zero_division=0),
        'recall': recall_score(y, prediction, zero_division=0),
        'f1': f1_score(y, prediction, zero_division=0),
        'auroc': roc_auc_score(y, probability),
        'auprc': average_precision_score(y, probability),
        'brier': brier_score_loss(y, probability),
        'ece': expected_calibration_error(y, probability, n_bins=10),
        'far': fp / max(fp + tn, 1),
        'alert_rate': (tp + fp) / max(tp + tn + fp + fn, 1),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def save_predictions(path, y, probability, threshold, metadata):
    frame = pd.DataFrame({
        'instance_id': np.arange(len(y), dtype=int),
        'y_true': np.asarray(y, dtype=int),
        'probability': np.asarray(probability, dtype=float),
        'prediction': (np.asarray(probability) >= threshold).astype(int),
    })
    for key, value in metadata.items():
        frame[key] = value
    temporary = path.with_suffix('.tmp.parquet')
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)


def write_manifest(datasets):
    manifest = {
        'benchmark': 'ER-CyRIS Algorithmic Diversity Benchmark v1',
        'datasets': {
            name: {'rows': int(len(y)), 'features': int(X.shape[1]), 'positive_rate': float(y.mean())}
            for name, (X, y, _) in datasets.items()
        },
        'models': MODEL_NAMES,
        'representations': REPRESENTATIONS,
        'conditions': CONDITIONS,
        'seeds': SEEDS,
        'max_samples_per_dataset': MAX_SAMPLES_PER_DATASET,
        'holdout_claim': 'untouched internal holdout; not external organizational validation',
    }
    temporary = MANIFEST_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    os.replace(temporary, MANIFEST_PATH)
    return manifest


In [24]:
def normalize_shap(values):
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    if values.ndim != 2:
        raise ValueError(f'Expected 2-D SHAP values, received {values.shape}')
    return values.astype(float)


def explain_model(model_name, model, calibrator, background, samples):
    if model_name in {'DecisionTree', 'RandomForest', 'XGBoost'}:
        explainer = shap.TreeExplainer(model)
        return normalize_shap(explainer.shap_values(samples))
    if model_name == 'LogisticRegression':
        explainer = shap.LinearExplainer(model, background)
        return normalize_shap(explainer.shap_values(samples))

    def calibrated_predict(matrix):
        return calibrator.predict_proba(raw_model_score(model, np.asarray(matrix)))

    explainer = shap.Explainer(calibrated_predict, background, algorithm='permutation')
    explanation = explainer(
        samples,
        max_evals=max(2 * samples.shape[1] + 1, 101),
        silent=True,
    )
    return normalize_shap(explanation.values)


def explanation_stability(clean_values, noisy_values, feature_names, top_k):
    k = min(int(top_k), clean_values.shape[1])
    clean_abs = np.abs(clean_values)
    noisy_abs = np.abs(noisy_values)
    clean_top = np.argsort(-clean_abs, axis=1)[:, :k]
    noisy_top = np.argsort(-noisy_abs, axis=1)[:, :k]
    jaccard = []
    for left, right in zip(clean_top, noisy_top):
        left_set, right_set = set(left.tolist()), set(right.tolist())
        jaccard.append(len(left_set & right_set) / max(len(left_set | right_set), 1))

    clean_global = clean_abs.mean(axis=0)
    noisy_global = noisy_abs.mean(axis=0)
    rank_rho = spearmanr(clean_global, noisy_global).statistic
    magnitude = 1.0 - np.mean(np.abs(clean_abs - noisy_abs)) / (np.mean(clean_abs) + 1e-9)
    sign_mask = (clean_values != 0) | (noisy_values != 0)
    sign_consistency = np.mean(np.sign(clean_values[sign_mask]) == np.sign(noisy_values[sign_mask]))
    global_top = np.argsort(-clean_global)[:k]
    return {
        'topk_jaccard_instance_mean': float(np.mean(jaccard)),
        'rank_spearman_rho': float(rank_rho),
        'magnitude_stability': float(np.clip(magnitude, 0.0, 1.0)),
        'sign_consistency': float(sign_consistency),
        'top_features_clean': json.dumps([feature_names[int(i)] for i in global_top]),
    }


def maybe_run_xai(dataset, representation, model_name, seed, model, calibrator,
                  X_train, X_development, feature_names):
    if not RUN_XAI or seed not in XAI_SEEDS:
        return
    xai_run_id = f'{dataset}__{representation}__{model_name}__seed-{seed}'
    existing = pd.read_csv(XAI_RESULTS) if XAI_RESULTS.exists() else pd.DataFrame()
    if not existing.empty and xai_run_id in set(existing['run_id'].astype(str)):
        return
    rng = np.random.default_rng(seed)
    sample_count = min(SHAP_N, len(X_development))
    background_count = min(SHAP_BACKGROUND_N, len(X_train))
    sample_indices = rng.choice(len(X_development), sample_count, replace=False)
    background_indices = rng.choice(len(X_train), background_count, replace=False)
    clean_samples = X_development[sample_indices]
    noisy_samples = add_gaussian_noise(clean_samples, sigma=0.05, seed=seed + 500)
    background = X_train[background_indices]
    started = time.perf_counter()
    try:
        clean_values = explain_model(model_name, model, calibrator, background, clean_samples)
        noisy_values = explain_model(model_name, model, calibrator, background, noisy_samples)
        row = {
            'run_id': xai_run_id,
            'dataset': dataset,
            'representation': representation,
            'model': model_name,
            'seed': seed,
            'status': 'succeeded',
            'shap_seconds': time.perf_counter() - started,
            **explanation_stability(clean_values, noisy_values, feature_names, TOP_K),
        }
    except Exception as error:
        row = {
            'run_id': xai_run_id,
            'dataset': dataset,
            'representation': representation,
            'model': model_name,
            'seed': seed,
            'status': 'failed',
            'error': repr(error),
            'shap_seconds': time.perf_counter() - started,
        }
    append_unique_row(XAI_RESULTS, row)


In [25]:
DATASETS = load_requested_datasets()
MANIFEST = write_manifest(DATASETS)
counts = expected_run_count(
    n_datasets=len(DATASETS),
    n_models=len(MODEL_NAMES),
    n_representations=len(REPRESENTATIONS),
    n_seeds=len(SEEDS),
    n_conditions=len(CONDITIONS),
)
print('Expected model fits:', counts['model_fits'])
print('Expected development metric rows:', counts['evaluation_rows'])
print('XAI runs:', len(DATASETS) * len(MODEL_NAMES) * len(REPRESENTATIONS) * len(XAI_SEEDS))


BGL: 30,000 rows, 11 features, anomaly=12.023%
HDFS: 30,000 rows, 8 features, anomaly=2.927%
UNSW-NB15: 30,000 rows, 48 features, anomaly=3.173%
Expected model fits: 180
Expected development metric rows: 360
XAI runs: 36


In [26]:
def run_development_benchmark(datasets):
    existing_results = pd.read_csv(DEVELOPMENT_RESULTS) if DEVELOPMENT_RESULTS.exists() else pd.DataFrame()
    completed = set(existing_results['run_id'].astype(str)) if not existing_results.empty else set()
    existing_thresholds = pd.read_csv(THRESHOLD_REGISTRY) if THRESHOLD_REGISTRY.exists() else pd.DataFrame()
    frozen_fits = set(existing_thresholds['fit_id'].astype(str)) if not existing_thresholds.empty else set()

    for dataset, (X, y, original_names) in datasets.items():
        for seed in SEEDS:
            split = make_four_way_split(y, seed=seed)
            for representation_name in REPRESENTATIONS:
                representation = REPRESENTATION_FACTORY[representation_name]().fit(X[split['train']])
                transformed = {
                    name: representation.transform(X[indices])
                    for name, indices in split.items()
                }
                feature_names = representation.feature_names(original_names)

                for model_name in MODEL_NAMES:
                    fit_id = f'{dataset}__{representation_name}__{model_name}__seed-{seed}'
                    expected_ids = {
                        build_run_id(dataset, representation_name, model_name, seed, condition, 'development')
                        for condition in CONDITIONS
                    }
                    xai_already_done = (
                        not RUN_XAI or seed not in XAI_SEEDS or
                        (XAI_RESULTS.exists() and fit_id in set(pd.read_csv(XAI_RESULTS)['run_id'].astype(str)))
                    )
                    if expected_ids.issubset(completed) and fit_id in frozen_fits and xai_already_done:
                        print('skip', fit_id)
                        continue

                    print('fit', fit_id)
                    model = make_model(model_name, seed)
                    rss_before = peak_rss_mb()
                    train_started = time.perf_counter()
                    model = fit_with_balancing(model, transformed['train'], y[split['train']], seed)
                    train_seconds = time.perf_counter() - train_started
                    rss_increment = max(0.0, peak_rss_mb() - rss_before)

                    calibrator = PlattCalibrator(seed=seed).fit(
                        raw_model_score(model, transformed['calibration']),
                        y[split['calibration']],
                    )
                    development_clean_probability = calibrator.predict_proba(
                        raw_model_score(model, transformed['development'])
                    )
                    threshold, development_f1 = select_f1_threshold(
                        y[split['development']], development_clean_probability
                    )
                    threshold_row = {
                        'fit_id': fit_id,
                        'dataset': dataset,
                        'representation': representation_name,
                        'model': model_name,
                        'seed': seed,
                        'threshold': threshold,
                        'development_clean_f1_at_selection': development_f1,
                    }
                    append_unique_row(THRESHOLD_REGISTRY, threshold_row, key='fit_id')
                    frozen_fits.add(fit_id)

                    for condition, sigma in CONDITIONS.items():
                        run_id = build_run_id(
                            dataset, representation_name, model_name, seed,
                            condition, 'development',
                        )
                        if run_id in completed:
                            continue
                        evaluation_X = add_gaussian_noise(
                            transformed['development'], sigma=sigma, seed=seed + 1000
                        )
                        inference_started = time.perf_counter()
                        probability = calibrator.predict_proba(raw_model_score(model, evaluation_X))
                        inference_seconds = time.perf_counter() - inference_started
                        metrics = classification_metrics(
                            y[split['development']], probability, threshold
                        )
                        row = {
                            'run_id': run_id,
                            'dataset': dataset,
                            'representation': representation_name,
                            'model': model_name,
                            'seed': seed,
                            'condition': condition,
                            'sigma': sigma,
                            'split': 'development',
                            'n_train': len(split['train']),
                            'n_calibration': len(split['calibration']),
                            'n_evaluation': len(split['development']),
                            'feature_count': evaluation_X.shape[1],
                            'threshold': threshold,
                            'train_seconds': train_seconds,
                            'inference_seconds': inference_seconds,
                            'throughput_rows_per_second': len(evaluation_X) / max(inference_seconds, 1e-9),
                            'peak_rss_increment_mb_approx': rss_increment,
                            **metrics,
                        }
                        append_unique_row(DEVELOPMENT_RESULTS, row)
                        completed.add(run_id)
                        save_predictions(
                            OUT_DIR / 'predictions_development' / f'{run_id}.parquet',
                            y[split['development']], probability, threshold,
                            {
                                'dataset': dataset, 'representation': representation_name,
                                'model': model_name, 'seed': seed, 'condition': condition,
                            },
                        )

                    maybe_run_xai(
                        dataset, representation_name, model_name, seed,
                        model, calibrator, transformed['train'],
                        transformed['development'], feature_names,
                    )
                    del model, calibrator
                    gc.collect()

    return pd.read_csv(DEVELOPMENT_RESULTS)


if RUN_DEVELOPMENT:
    DEVELOPMENT_FRAME = run_development_benchmark(DATASETS)
    print('Development rows:', len(DEVELOPMENT_FRAME))
else:
    print('RUN_DEVELOPMENT=False; benchmark not executed')


skip BGL__raw_telemetry_space__LogisticRegression__seed-42
skip BGL__raw_telemetry_space__DecisionTree__seed-42
skip BGL__raw_telemetry_space__RandomForest__seed-42
skip BGL__raw_telemetry_space__XGBoost__seed-42
skip BGL__raw_telemetry_space__SVC_RBF__seed-42
skip BGL__raw_telemetry_space__MLP__seed-42
skip BGL__transformed_feature_space__LogisticRegression__seed-42
skip BGL__transformed_feature_space__DecisionTree__seed-42
skip BGL__transformed_feature_space__RandomForest__seed-42
skip BGL__transformed_feature_space__XGBoost__seed-42
skip BGL__transformed_feature_space__SVC_RBF__seed-42
skip BGL__transformed_feature_space__MLP__seed-42
skip BGL__raw_telemetry_space__LogisticRegression__seed-52
skip BGL__raw_telemetry_space__DecisionTree__seed-52
skip BGL__raw_telemetry_space__RandomForest__seed-52
skip BGL__raw_telemetry_space__XGBoost__seed-52
skip BGL__raw_telemetry_space__SVC_RBF__seed-52
skip BGL__raw_telemetry_space__MLP__seed-52
skip BGL__transformed_feature_space__LogisticRegr

## Bekukan threshold sebelum holdout

Cell berikut hanya boleh dijalankan setelah seluruh kombinasi development selesai.
Digest mengikat threshold terhadap dataset, representasi, model, dan seed. Perubahan
threshold setelah holdout merupakan pelanggaran protokol.


In [27]:
FREEZE_PATH = OUT_DIR / 'FROZEN_THRESHOLD_REGISTRY.json'
threshold_frame = pd.read_csv(THRESHOLD_REGISTRY)
expected_thresholds = len(DATASETS) * len(REPRESENTATIONS) * len(MODEL_NAMES) * len(SEEDS)
if len(threshold_frame) != expected_thresholds:
    raise RuntimeError(
        f'Threshold registry belum lengkap: {len(threshold_frame)}/{expected_thresholds}'
    )
threshold_records = threshold_frame.to_dict(orient='records')
threshold_digest = freeze_registry_digest(threshold_records)
freeze_payload = {
    'status': 'FROZEN',
    'threshold_rows': len(threshold_frame),
    'sha256': threshold_digest,
    'rule': 'selected on clean development only; never revised from holdout',
}
if FREEZE_PATH.exists():
    previous = json.loads(FREEZE_PATH.read_text(encoding='utf-8'))
    if previous['sha256'] != threshold_digest:
        raise RuntimeError('Threshold registry berubah setelah freeze. Stop analysis.')
else:
    temporary = FREEZE_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(freeze_payload, indent=2), encoding='utf-8')
    os.replace(temporary, FREEZE_PATH)
print(json.dumps(freeze_payload, indent=2))


{
  "status": "FROZEN",
  "threshold_rows": 180,
  "sha256": "b70a7a4dd26b42c9a5376d45c781cdf389e76827fbaecc077520fe7437b2bb30",
  "rule": "selected on clean development only; never revised from holdout"
}


In [28]:
def holm_adjust(p_values):
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running = 0.0
    total = len(p_values)
    for rank, index in enumerate(order):
        value = min(1.0, (total - rank) * p_values[index])
        running = max(running, value)
        adjusted[index] = running
    return adjusted


def model_comparison(frame, metric):
    block_columns = ['dataset', 'representation', 'seed', 'condition']
    pivot = frame.pivot_table(index=block_columns, columns='model', values=metric, aggfunc='mean')
    pivot = pivot.dropna(subset=MODEL_NAMES)
    if pivot.empty:
        raise RuntimeError(f'No complete blocks for {metric}')
    friedman = friedmanchisquare(*(pivot[name].to_numpy() for name in MODEL_NAMES))
    rows = []
    for left, right in itertools.combinations(MODEL_NAMES, 2):
        try:
            test = wilcoxon(pivot[left], pivot[right], zero_method='wilcox')
            statistic, p_value = float(test.statistic), float(test.pvalue)
        except ValueError:
            statistic, p_value = 0.0, 1.0
        rows.append({
            'metric': metric, 'model_a': left, 'model_b': right,
            'wilcoxon_statistic': statistic, 'p_raw': p_value,
        })
    pairwise = pd.DataFrame(rows)
    pairwise['p_holm'] = holm_adjust(pairwise['p_raw'].to_numpy())
    pairwise['significant_0p05'] = pairwise['p_holm'] < 0.05
    return {
        'metric': metric,
        'friedman_statistic': float(friedman.statistic),
        'friedman_p': float(friedman.pvalue),
        'complete_blocks': int(len(pivot)),
    }, pairwise


development = pd.read_csv(DEVELOPMENT_RESULTS)
summary = development.groupby(
    ['dataset', 'representation', 'model', 'condition'], as_index=False
).agg(
    f1_mean=('f1', 'mean'), f1_sd=('f1', 'std'),
    auprc_mean=('auprc', 'mean'), brier_mean=('brier', 'mean'),
    ece_mean=('ece', 'mean'), train_seconds_mean=('train_seconds', 'mean'),
    throughput_mean=('throughput_rows_per_second', 'mean'),
    memory_mb_mean=('peak_rss_increment_mb_approx', 'mean'),
)
atomic_csv(summary, OUT_DIR / 'algorithmic_diversity_summary.csv')

omnibus_rows, pairwise_frames = [], []
for metric in ['f1', 'auprc', 'brier', 'ece']:
    omnibus, pairwise = model_comparison(development, metric)
    omnibus_rows.append(omnibus)
    pairwise_frames.append(pairwise)
atomic_csv(pd.DataFrame(omnibus_rows), OUT_DIR / 'friedman_results.csv')
atomic_csv(pd.concat(pairwise_frames, ignore_index=True), OUT_DIR / 'pairwise_wilcoxon_holm.csv')

if XAI_RESULTS.exists():
    xai = pd.read_csv(XAI_RESULTS)
    succeeded = xai[xai['status'] == 'succeeded'].copy()
    atomic_csv(succeeded, OUT_DIR / 'xai_stability_successful.csv')
    print('XAI succeeded:', len(succeeded), '/', len(xai))
display(summary.head(30))
display(pd.DataFrame(omnibus_rows))


XAI succeeded: 36 / 36


,dataset,representation,model,condition,f1_mean,f1_sd,auprc_mean,brier_mean,ece_mean,train_seconds_mean,throughput_mean,memory_mb_mean
0,BGL,raw_telemetry_space,DecisionTree,clean,0.996123,0.002103,0.994330,0.001027,0.000836,0.014410,2.905999e+06,0.0
1,BGL,raw_telemetry_space,DecisionTree,noise_sigma_0p05,0.247810,0.056454,0.219434,0.110513,0.108834,0.014410,3.698246e+06,0.0
2,BGL,raw_telemetry_space,LogisticRegression,clean,0.966932,0.003420,0.832497,0.019414,0.057954,0.029708,4.057448e+06,0.0
3,BGL,raw_telemetry_space,LogisticRegression,noise_sigma_0p05,0.713448,0.013395,0.686582,0.049052,0.022760,0.029708,4.804057e+06,0.0
4,BGL,raw_telemetry_space,MLP,clean,0.993567,0.002230,0.979509,0.001905,0.001388,9.371957,3.571198e+05,0.0
5,BGL,raw_telemetry_space,MLP,noise_sigma_0p05,0.741416,0.057648,0.840041,0.045704,0.040124,9.371957,4.231150e+05,0.0
6,BGL,raw_telemetry_space,RandomForest,clean,0.997046,0.001520,0.998501,0.000821,0.001283,1.585899,4.338513e+04,0.0
7,BGL,raw_telemetry_space,RandomForest,noise_sigma_0p05,0.270939,0.037774,0.548875,0.087728,0.081737,1.585899,4.275941e+04,0.0
8,BGL,raw_telemetry_space,SVC_RBF,clean,0.990838,0.008223,0.952158,0.006235,0.010161,1.558505,9.352694e+03,0.0
9,BGL,raw_telemetry_space,SVC_RBF,noise_sigma_0p05,0.608957,0.022213,0.821133,0.048923,0.042899,1.558505,8.963565e+03,0.0


,metric,friedman_statistic,friedman_p,complete_blocks
0,f1,45.621998,1.084233e-08,60
1,auprc,121.466667,1.534738e-24,60
2,brier,54.133333,1.967662e-10,60
3,ece,25.676190,1.031088e-04,60


## Untouched internal holdout

Ini bukan external validation. External validation memerlukan sumber data atau
organisasi yang benar-benar independen. Holdout di bawah hanya menguji apakah
hasil development bertahan pada partisi yang tidak digunakan untuk training,
calibration, maupun threshold selection.


In [29]:
HOLDOUT_RESULTS = OUT_DIR / 'holdout_results.csv'
HOLDOUT_LOCK = OUT_DIR / 'HOLDOUT_EVALUATION_LOCK.json'


def run_untouched_holdout(datasets):
    if not RUN_UNTOUCHED_HOLDOUT:
        print('Holdout terkunci. Tinjau hasil development lalu set RUN_UNTOUCHED_HOLDOUT=True.')
        return None
    if not FREEZE_PATH.exists():
        raise RuntimeError('Threshold belum dibekukan.')
    freeze_payload = json.loads(FREEZE_PATH.read_text(encoding='utf-8'))
    if HOLDOUT_LOCK.exists():
        lock = json.loads(HOLDOUT_LOCK.read_text(encoding='utf-8'))
        if lock.get('status') == 'COMPLETE':
            raise RuntimeError('Holdout sudah selesai dan tidak boleh dijalankan ulang.')
        if lock.get('threshold_sha256') != freeze_payload['sha256']:
            raise RuntimeError('Threshold digest berbeda dari holdout lock.')
    else:
        HOLDOUT_LOCK.write_text(json.dumps({
            'status': 'IN_PROGRESS',
            'threshold_sha256': freeze_payload['sha256'],
        }, indent=2), encoding='utf-8')

    registry = pd.read_csv(THRESHOLD_REGISTRY).set_index('fit_id')
    current = pd.read_csv(HOLDOUT_RESULTS) if HOLDOUT_RESULTS.exists() else pd.DataFrame()
    completed = set(current['run_id'].astype(str)) if not current.empty else set()

    for dataset, (X, y, original_names) in datasets.items():
        for seed in SEEDS:
            split = make_four_way_split(y, seed=seed)
            for representation_name in REPRESENTATIONS:
                representation = REPRESENTATION_FACTORY[representation_name]().fit(X[split['train']])
                transformed = {
                    name: representation.transform(X[indices])
                    for name, indices in split.items()
                }
                for model_name in MODEL_NAMES:
                    fit_id = f'{dataset}__{representation_name}__{model_name}__seed-{seed}'
                    threshold = float(registry.loc[fit_id, 'threshold'])
                    expected_ids = {
                        build_run_id(dataset, representation_name, model_name, seed, condition, 'holdout')
                        for condition in CONDITIONS
                    }
                    if expected_ids.issubset(completed):
                        continue
                    model = fit_with_balancing(
                        make_model(model_name, seed),
                        transformed['train'], y[split['train']], seed,
                    )
                    calibrator = PlattCalibrator(seed=seed).fit(
                        raw_model_score(model, transformed['calibration']),
                        y[split['calibration']],
                    )
                    for condition, sigma in CONDITIONS.items():
                        run_id = build_run_id(
                            dataset, representation_name, model_name, seed,
                            condition, 'holdout',
                        )
                        if run_id in completed:
                            continue
                        evaluation_X = add_gaussian_noise(
                            transformed['holdout'], sigma=sigma, seed=seed + 2000
                        )
                        started = time.perf_counter()
                        probability = calibrator.predict_proba(raw_model_score(model, evaluation_X))
                        inference_seconds = time.perf_counter() - started
                        row = {
                            'run_id': run_id, 'dataset': dataset,
                            'representation': representation_name, 'model': model_name,
                            'seed': seed, 'condition': condition, 'sigma': sigma,
                            'split': 'untouched_internal_holdout',
                            'n_evaluation': len(evaluation_X),
                            'feature_count': evaluation_X.shape[1],
                            'threshold': threshold,
                            'inference_seconds': inference_seconds,
                            'throughput_rows_per_second': len(evaluation_X) / max(inference_seconds, 1e-9),
                            **classification_metrics(y[split['holdout']], probability, threshold),
                        }
                        append_unique_row(HOLDOUT_RESULTS, row)
                        completed.add(run_id)
                        save_predictions(
                            OUT_DIR / 'predictions_holdout' / f'{run_id}.parquet',
                            y[split['holdout']], probability, threshold,
                            {
                                'dataset': dataset, 'representation': representation_name,
                                'model': model_name, 'seed': seed, 'condition': condition,
                            },
                        )
                    del model, calibrator
                    gc.collect()

    expected_rows = len(datasets) * len(REPRESENTATIONS) * len(MODEL_NAMES) * len(SEEDS) * len(CONDITIONS)
    result = pd.read_csv(HOLDOUT_RESULTS)
    if len(result) == expected_rows:
        temporary = HOLDOUT_LOCK.with_suffix('.tmp')
        temporary.write_text(json.dumps({
            'status': 'COMPLETE',
            'threshold_sha256': freeze_payload['sha256'],
            'result_rows': len(result),
        }, indent=2), encoding='utf-8')
        os.replace(temporary, HOLDOUT_LOCK)
    return result


HOLDOUT_FRAME = run_untouched_holdout(DATASETS)
if HOLDOUT_FRAME is not None:
    print('Holdout rows:', len(HOLDOUT_FRAME))


Holdout rows: 360


## Interpretasi dan pekerjaan berikutnya

Notebook ini menghasilkan bukti untuk **algorithm variety**, sebagian
**data variety**, serta snapshot awal **cost dan latency**. Hasilnya digunakan
untuk memilih model representatif dan menyusun pengujian berikutnya:

1. detector-only;
2. detector + XAI tanpa reliability gate;
3. full ER-CyRIS dengan `accept`, `warn`, dan `abstain`;
4. threshold sensitivity tanpa mengubah frozen threshold utama;
5. scalability curve terhadap jumlah observasi dan jumlah fitur;
6. decision validity menggunakan outcome risiko atau penilaian pakar.

**Prohibited claims:** model-agnostic universal, external organizational
validation, real-time readiness, dan validasi M5-M6 tidak boleh dinyatakan
hanya dari notebook ini.
